# Models v bookmakers: the long backtest (2019/20 to today)

Companion notebook to the **Track record** page of [Jan's Matchday](https://jhvl88.github.io/matchday/record/).
It replays every Bundesliga match since 2019/20 as if it were next weekend's fixture and grades the three
published models against **one** benchmark: the market-average closing price.

1. **Source data** straight from football-data.co.uk, checked file by file against the checksums of the published run.
2. **Posts' data check**: every 2026/27 number embedded in the published posts' notebooks, compared with the source.
3. **Walk-forward backtest**: Dixon–Coles refit weekly, XGBoost refit per season, the Bayesian shots-to-goals model refit monthly (NUTS on GPU).
4. **Grades**: ranked probability score (RPS), log loss and top-pick rate, the gap to the market with a bootstrap 95% interval, season by season.

**Runtime:** about 10 minutes with `RUN_BAYES = False`, which loads the Bayesian predictions from the published run.
`RUN_BAYES = True` refits the Bayesian model about 75 times on your GPU (Runtime › Change runtime type › GPU); expect an hour or more.

The model code lives in [`analysis/pipeline/`](https://github.com/JHvL88/matchday/tree/main/analysis/pipeline), the same files the posts use.

In [ ]:
!nvidia-smi -L || echo "No GPU found: fine unless RUN_BAYES = True"

In [ ]:
# ---------------------------------------------------------------- settings
RUN_BAYES = False                  # True = refit the Bayesian model monthly on this machine
FIRST_SEASON, LAST_SEASON = 2019, 2026
REPO, BRANCH = "https://github.com/JHvL88/matchday", "main"
RESULTS_COMMIT = "3c69dd91191235deb8481cd84b9f81518d251a5e"   # backtest-results commit behind the published figures

import os, sys, json, glob, subprocess, urllib.request
if not os.path.exists("matchday"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, "matchday"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpyro", "xgboost"], check=True)
sys.path.insert(0, "matchday/analysis/pipeline")
PIPE = "matchday/analysis/pipeline"
os.makedirs("results", exist_ok=True)

## 1. Source data, straight from football-data.co.uk

The fetcher saves each season file exactly as served, with a SHA-256 checksum. Finished seasons should be
byte-identical to the files the published run used; the current season's file grows every week.
If football-data.co.uk can't be reached, the cell falls back to the stored copy on the repo's `source-data` branch.

In [ ]:
fetched = subprocess.run([sys.executable, f"{PIPE}/fetch_football_data.py", "--out", "source", "--from", "2005"]).returncode == 0
if not fetched:
    print("\nfootball-data.co.uk not reachable from here: using the stored copy (source-data branch).")
    subprocess.run(["rm", "-rf", "source"])
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "source-data", REPO, "source"], check=True)

published = {f["path"]: f["sha256"] for f in json.load(open("matchday/analysis/backtest/source_manifest.json"))["files"]}
mine = {f["path"]: f["sha256"] for f in json.load(open("source/manifest.json"))["files"]}
same = sorted(p for p in mine if published.get(p) == mine[p])
changed = sorted(p for p in mine if p in published and published[p] != mine[p])
print(f"{len(same)} of {len(mine)} files byte-identical to the published run.")
print("Changed since then:", changed or "none")

## 2. Check the numbers behind the published posts

Each post's notebook carries the 2026/27 rows it used. They were transcribed, so every cell is compared with the source file.

In [ ]:
from verify_sources import check_embedded
for nb in sorted(glob.glob("matchday/analysis/*/*.ipynb")):
    r = check_embedded(nb, "source")
    if r is None:
        continue
    print(f"{nb.split('/')[2]}: {r['rows']} rows, {r['cells']} cells, {r['mismatched_cells']} differ")
    for d in r["mismatches"]:
        print("   ", d)

## 3. One match table, built from the source files

1. and 2. Bundesliga since 2005/06. Every row keeps its source file and line. The benchmark is `avgc_*`:
the market-average **closing** price, which football-data.co.uk records for every match since 2019/20.

In [ ]:
!python {PIPE}/build_matches.py source data/matches.csv

## 4. Walk-forward backtest

Every forecast uses only matches played before it. Dixon–Coles' time decay is tuned on 2018/19, the season
before the window, so no tuning choice has seen a test match. XGBoost early-stops on the last training season.

In [ ]:
!python {PIPE}/backtest.py dc  data/matches.csv results/oos_dc.csv  --seasons {FIRST_SEASON} {LAST_SEASON}
!python {PIPE}/backtest.py xgb data/matches.csv results/oos_xgb.csv --seasons {FIRST_SEASON} {LAST_SEASON}

In [ ]:
if RUN_BAYES:
    for s in range(FIRST_SEASON, LAST_SEASON + 1):
        subprocess.run([sys.executable, f"{PIPE}/backtest.py", "bayes", "data/matches.csv",
                        f"results/oos_bay_{s}.csv", "--seasons", str(s), str(s)], check=True)
else:
    base = f"https://raw.githubusercontent.com/JHvL88/matchday/{RESULTS_COMMIT}/latest/"
    for s in range(FIRST_SEASON, LAST_SEASON + 1):
        urllib.request.urlretrieve(base + f"oos_bay_{s}.csv", f"results/oos_bay_{s}.csv")
    print("Bayesian predictions loaded from the published run:", base)

## 5. Grades

**RPS** (ranked probability score) measures how far the home/draw/away probabilities were from what happened,
respecting the order home > draw > away. Lower is better; one-third for everything scores about 0.23.
The **gap** is the ensemble's RPS minus the market's, so positive means the bookmakers were better.
The interval comes from resampling matches 4,000 times.

In [ ]:
!python {PIPE}/evaluate.py data/matches.csv results results/results.json

In [ ]:
!python {PIPE}/make_figures.py results/results.json figs
from IPython.display import SVG, display
for f in ["gap_by_season", "bridge"]:
    if os.path.exists(f"figs/{f}.svg"):
        display(SVG(f"figs/{f}.svg"))

### Same answer as the published run?

Dixon–Coles is deterministic, and XGBoost is deterministic on the same hardware (a GPU can shift it in the fourth decimal).
Bayesian refits on your machine draw different random samples, so expect small differences when `RUN_BAYES = True`.

In [ ]:
pub = json.load(open("matchday/analysis/backtest/results.json"))
new = json.load(open("results/results.json"))
for m in ["avgc", "ens", "dc", "bay", "xgb"]:
    print(f"{m:5s} published RPS {pub['rows'][m]['rps']:.4f}   this run {new['rows'][m]['rps']:.4f}")
print(f"gap   published {pub['gap']['gap']:+.4f}   this run {new['gap']['gap']:+.4f}")